# MovieLens-32M 生成式推荐 — Colab 训练笔记本

本 notebook 会在 Colab 上完成完整训练流程：

1. 拉取项目代码（GitHub 克隆 / 或挂载 Google Drive）
2. 安装依赖（Colab 自带的 torch/transformers 通常已经够用）
3. 下载 MovieLens-32M 数据集
4. 跑 Stage 1 ~ Stage 4（预处理 → RQ-VAE → 序列生成 → TIGER 训练 → 评测）
5. 把权重和输出回写到 Google Drive（可选）

**首先把 Colab 切到 GPU runtime**：菜单 `Runtime → Change runtime type → Hardware accelerator → T4 GPU`。

> 笔记本里所有 cell 都可以从上到下顺序执行。如果运行时断掉，重新挂载 Drive 后从「环境检查」cell 接着跑即可。

## 0. 配置项（按需修改）

- 默认会从 `GITHUB_URL` 克隆代码。
- 如果你更倾向于把代码先上传到 Google Drive 再用，把 `USE_DRIVE_FOR_CODE` 改为 `True` 并设置 `DRIVE_PROJECT_PATH`。
- `DRIVE_PERSIST_PATH` 用来在训练结束后把 `models/` 和 `outputs/` 备份到 Drive，避免 Colab 实例回收后丢失。

In [ ]:
GITHUB_URL = 'https://github.com/MasterpieceXu/GR-movie-recommendation.git'
GIT_BRANCH = 'main'

USE_DRIVE_FOR_CODE = False
DRIVE_PROJECT_PATH = '/content/drive/MyDrive/GR-movie-recommendation'

PERSIST_TO_DRIVE = True
DRIVE_PERSIST_PATH = '/content/drive/MyDrive/GR-movie-recommendation-runs'

STAGES = '0,1,2,3,4'  # 想训练 OneRec-lite 时改成 '0,1,2,3,4,5'

PROJECT_DIR = '/content/GR-movie-recommendation'

## 1. 取得项目代码

In [ ]:
import os, subprocess, shutil

if USE_DRIVE_FOR_CODE or PERSIST_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

if USE_DRIVE_FOR_CODE:
    assert os.path.isdir(DRIVE_PROJECT_PATH), f'{DRIVE_PROJECT_PATH} 不存在，请先把项目上传到这里'
    if os.path.exists(PROJECT_DIR):
        shutil.rmtree(PROJECT_DIR)
    shutil.copytree(DRIVE_PROJECT_PATH, PROJECT_DIR)
else:
    if os.path.exists(PROJECT_DIR):
        print(f'{PROJECT_DIR} 已存在，跳过 clone')
    else:
        subprocess.check_call(['git', 'clone', '--branch', GIT_BRANCH, GITHUB_URL, PROJECT_DIR])

%cd $PROJECT_DIR
!ls

## 2. 安装依赖

Colab 自带的 `torch` 已经装好 CUDA 版本，不会被 `requirements.txt` 顶掉（pip 会自动跳过已满足的版本约束）。第一次安装大概 30~60 秒。

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import torch, transformers, sys, platform
print('python      :', platform.python_version())
print('torch       :', torch.__version__, '| cuda:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')
print('transformers:', transformers.__version__)
assert torch.cuda.is_available(), 'GPU 没开，请把 runtime 改成 GPU 后重试'

## 3. 下载 MovieLens-32M

数据约 240MB，解压后 ~1GB。如果之前下过并放在 Drive 里，可以跳过这步，把数据软链到 `dataset/ml-32m`。

In [ ]:
import os, subprocess
DATA_DIR = 'dataset'
TARGET = os.path.join(DATA_DIR, 'ml-32m')
os.makedirs(DATA_DIR, exist_ok=True)

if not os.path.exists(os.path.join(TARGET, 'ratings.csv')):
    if not os.path.exists('ml-32m.zip'):
        !wget -q https://files.grouplens.org/datasets/movielens/ml-32m.zip
    !unzip -q -o ml-32m.zip -d $DATA_DIR
    print('解压完成')
else:
    print('检测到已有 ratings.csv，跳过下载')

!ls -lh $TARGET

## 4. 训练前的小贴士（可选：调小 batch / 步数做冒烟测试）

如果想先跑一遍流程确认环境无误，可以临时把 epochs 调小。把下面 cell 的 `SMOKE_TEST` 改成 `True` 即可在 ~10 分钟内跑完一遍 Stage 1~3。

In [ ]:
SMOKE_TEST = False

if SMOKE_TEST:
    import config as _cfg
    _cfg.RQVAEConfig.epochs = 2
    _cfg.TIGERConfig.num_train_epochs = 1
    _cfg.TIGERConfig.eval_steps = 50
    _cfg.TIGERConfig.save_steps = 200
    print('SMOKE_TEST 已启用，缩短了训练步数')

## 5. 跑完整 pipeline

`scripts/run_pipeline.py` 会按 `STAGES` 配置顺序执行：

- **Stage 0**：环境检查
- **Stage 1**：数据预处理 + RQ-VAE 训练 + 语义 ID 生成
- **Stage 2**：用户序列生成
- **Stage 3**：TIGER（T5-small 微调）训练
- **Stage 4**：离线评测
- **Stage 5**（可选）：OneRec-lite 多项目生成 + DPO

在 T4 GPU 上完整跑一次 Stage 1~4 大约需要 2~3 小时。

In [ ]:
!python scripts/run_pipeline.py --stages $STAGES

## 6. 把模型 / 输出备份到 Google Drive

Colab 的 `/content` 目录是 ephemeral 的，实例回收后会丢。建议把 `models/`、`outputs/`、`logs/` 三个目录复制一份到 Drive。

In [ ]:
import os, shutil, datetime
if PERSIST_TO_DRIVE:
    stamp = datetime.datetime.now().strftime('%Y%m%d-%H%M%S')
    target = os.path.join(DRIVE_PERSIST_PATH, f'run-{stamp}')
    os.makedirs(target, exist_ok=True)
    for sub in ('models', 'outputs', 'logs'):
        src = os.path.join(PROJECT_DIR, sub)
        if os.path.isdir(src):
            shutil.copytree(src, os.path.join(target, sub), dirs_exist_ok=True)
    print(f'已备份到: {target}')
else:
    print('PERSIST_TO_DRIVE=False，跳过备份')

## 7. 简单试用一下推荐结果（可选）

In [ ]:
import os, sys, torch
sys.path.insert(0, PROJECT_DIR)
from src.tiger_model import TIGERModel

model_path = os.path.join(PROJECT_DIR, 'models', 'tiger_final')
if os.path.isdir(model_path):
    model = TIGERModel.from_pretrained(model_path).to('cuda').eval()
    sample_seq = ['<id_1>', '<id_42>', '<id_7>', '<id_19>']
    print('Top-5 candidates:', model.recommend(sample_seq, num_recommendations=5, num_beams=10))
else:
    print('未找到 tiger_final，可能是 Stage 3 没跑成功')